# Test de personalidad
## 03 · Variables nuevas a partir de las respuestas de texto

Proyecto de Sistemas Inteligentes · UNAM · Matemáticas Aplicadas y Computación

## Objetivo

El EDA mostró que las variables numéricas y categóricas que ya teníamos casi no separan a los dos perfiles. Pero el formulario tiene muchas respuestas de **texto libre** que sí pueden aportar información.

En este cuaderno convertimos algunas de esas respuestas en **variables sencillas (0 o 1)** usando **reglas escritas y expresiones regulares**, sin IA ni NLP, como pide la maestra. Al final guardamos un archivo listo para el modelado.

## Datos

Usamos `Datos_Con_Perfil.xlsx` (el que generó el cuaderno 01). Si no está en Colab, la celda de carga te pedirá subirlo.

## Cómo se crean las variables

Todas las reglas buscan **palabras clave** en la respuesta. Cuando hay varias, **gana la que aparece primero** (lo que la persona dice primero es lo que más pesa). Si no aparece ninguna, la respuesta queda como "otro": **no inventamos una interpretación**.

| Variable nueva | Pregunta | Vale 1 cuando... |
|---|---|---|
| `rol_lider` | 10. Papel en el equipo | La respuesta empieza diciendo que es líder (en las primeras 5 palabras) |
| `reaccion_incomodidad` | 11. Cambio de planes | Lo primero que menciona es molestia, enojo, frustración, ansiedad o similar |
| `problema_con_otros` | 9. Problema que no sabe resolver | Lo primero que menciona es pedir ayuda, preguntar o consultar a otras personas |
| `act_conversar` | 13. Actividades con otras personas | Menciona platicar, hablar, conversar, compartir, convivir o anécdotas |
| `usa_spotify` | Música, pregunta 4 | Menciona Spotify |
| `usa_youtube` | Música, pregunta 4 | Menciona YouTube o YT (incluye YouTube Music) |

Además preparamos tres variables que ya existían: `genero_femenino`, `mus_tiempo_horas` (el mínimo de horas de escucha) y `tiene_pareja`.

**Importante:** las preguntas 6 y 7 **no** se usan aquí, porque con ellas se creó el perfil.

### Preparar el entorno
*Qué hace:* importa las herramientas y revisa si estamos en Google Colab.

In [ ]:
import os
import re
import unicodedata

import numpy as np
import pandas as pd

try:
    from google.colab import files  # solo existe dentro de Google Colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

print("pandas", pd.__version__, "| ejecutando en Colab:", EN_COLAB)

### Cargar los datos
*Qué hace:* busca `Datos_Con_Perfil.xlsx` (si no está, te pide subirlo) y lo lee.

In [ ]:
ARCHIVO = "Datos_Con_Perfil.xlsx"

if not os.path.exists(ARCHIVO):
    if EN_COLAB:
        print("Elige el archivo Datos_Con_Perfil.xlsx en la ventana que se abre...")
        files.upload()
    if not os.path.exists(ARCHIVO):
        raise FileNotFoundError(
            f"No se encontró '{ARCHIVO}'. Súbelo al panel de archivos de Colab con ese nombre exacto."
        )

df = pd.read_excel(ARCHIVO, sheet_name="DatosConPerfil")
print(f"Personas: {df.shape[0]} | Columnas: {df.shape[1]}")

### Funciones con las reglas
*Qué hace:* define cada regla como una función. Todas pasan el texto a minúsculas y sin acentos antes de buscar palabras clave, para que "Líder", "lider" y "LÍDER" cuenten igual.

In [ ]:
def normalizar(texto):
    """Minúsculas y sin acentos."""
    t = unicodedata.normalize("NFKD", str(texto))
    t = "".join(c for c in t if not unicodedata.combining(c))
    return t.lower()


def primeras_palabras(texto, n):
    return " ".join(texto.split()[:n])


def clasificar_rol(texto):
    """Pregunta 10: papel en el equipo."""
    if pd.isna(texto):
        return "sin_dato"
    t = normalizar(texto)
    if re.search(r"\bdepende", primeras_palabras(t, 3)):
        return "depende"
    if re.search(r"\blider", primeras_palabras(t, 5)):
        return "lider"
    m = re.search(r"organiz|colabor|desarrollador|ayud|aport|idea", t)
    if not m:
        return "otro"
    return "organizador" if m.group(0) == "organiz" else "colaborador"


INCOMODIDAD = r"enoj|moles|frustr|disgust|decepcion|ansiedad|inquiet|estres"
ADAPTACION = r"adapt|tranquil|fluy"


def clasificar_reaccion(texto):
    """Pregunta 11: reacción ante un cambio de planes."""
    if pd.isna(texto):
        return "sin_dato"
    t = normalizar(texto)
    m = re.search(f"{INCOMODIDAD}|{ADAPTACION}", t)
    if not m:
        return "otro"
    return "adaptacion" if m.group(0) in ("adapt", "tranquil", "fluy") else "incomodidad"


CON_OTROS = r"\bpid|\bpregunt|\bconsult|\bacud|\bayuda|\bopinion|\bconsejo|\bhabl"
POR_CUENTA = r"\binvestig|\bgoogle|\bia\b|\binternet|\byoutube"


def clasificar_problema(texto):
    """Pregunta 9: qué hace ante un problema que no sabe resolver."""
    if pd.isna(texto):
        return "sin_dato"
    t = normalizar(texto)
    a, b = re.search(CON_OTROS, t), re.search(POR_CUENTA, t)
    if a is None and b is None:
        return "otro"
    if a is not None and (b is None or a.start() < b.start()):
        return "con_otros"
    return "por_su_cuenta"


def menciona(texto, patron):
    """1 si el texto contiene el patrón, 0 si no, vacío si no hay respuesta."""
    if pd.isna(texto):
        return np.nan
    return int(bool(re.search(patron, normalizar(texto))))

### Aplicar las reglas
*Qué hace:* crea las categorías y las variables 0/1 para cada persona.

In [ ]:
df["rol_equipo"] = df["pers_q10"].map(clasificar_rol)
df["reaccion_cambio"] = df["pers_q11"].map(clasificar_reaccion)
df["problema_resuelve"] = df["pers_q09"].map(clasificar_problema)

df["rol_lider"] = (df["rol_equipo"] == "lider").astype(int)
df["reaccion_incomodidad"] = (df["reaccion_cambio"] == "incomodidad").astype(int)
df["problema_con_otros"] = (df["problema_resuelve"] == "con_otros").astype(int)
df["act_conversar"] = df["pers_q13"].map(lambda x: menciona(x, r"\bplatic|\bhabl|\bconvers|\bcompart|\bconviv|\banecdot|\bcharl"))
df["usa_spotify"] = df["mus_q04"].map(lambda x: menciona(x, r"spotify"))
df["usa_youtube"] = df["mus_q04"].map(lambda x: menciona(x, r"youtube|\byt\b"))

# Variables que ya existían, en formato 0/1 o numérico
df["genero_femenino"] = (df["pers_genero"] == "Femenino").astype(int)
df["mus_tiempo_horas"] = pd.to_numeric(df["mus_tiempo_min_horas"], errors="coerce")
df["tiene_pareja"] = df["amor_pareja"].map({"Sí": 1, "No": 0})  # vacío si no respondió ese formulario

for col in ["rol_equipo", "reaccion_cambio", "problema_resuelve"]:
    print(f"--- {col} ---")
    print(df[col].value_counts().to_dict())

### Revisar la clasificación persona por persona
*Qué hace:* muestra, para cada pregunta, la respuesta (recortada) y la categoría que le asignó la regla. **Hay que leerlas con calma**: si alguna asignación no tiene sentido, hay que corregir la regla y no el dato.

In [ ]:
def mostrar(col_texto, col_categoria):
    t = df[["id_persona", col_texto, col_categoria]].copy()
    t[col_texto] = t[col_texto].astype(str).str.slice(0, 70)
    display(t)

mostrar("pers_q10", "rol_equipo")

In [ ]:
mostrar("pers_q11", "reaccion_cambio")

In [ ]:
mostrar("pers_q09", "problema_resuelve")

In [ ]:
t = df[["id_persona", "pers_q13", "act_conversar", "mus_q04", "usa_spotify", "usa_youtube"]].copy()
t["pers_q13"] = t["pers_q13"].astype(str).str.slice(0, 45)
t["mus_q04"] = t["mus_q04"].astype(str).str.slice(0, 40)
display(t)

### Guardar el archivo para el modelado
*Qué hace:* guarda `Datos_Modelado.xlsx` con el perfil y las 9 variables que usaremos para predecir, más una hoja con las reglas. Las variables que se usaron para crear el perfil (preguntas 6 y 7) **no** están incluidas.

In [ ]:
VARIABLES = [
    "genero_femenino", "mus_tiempo_horas", "tiene_pareja",
    "rol_lider", "reaccion_incomodidad", "problema_con_otros",
    "act_conversar", "usa_spotify", "usa_youtube",
]

salida = df[["id_persona", "perfil", "perfil_origen"] + VARIABLES].copy()

print("Personas con valor 1 en cada variable:")
print(salida[[v for v in VARIABLES if v != "mus_tiempo_horas"]].sum().astype(int).to_string())
print("\nFaltantes por variable:")
print(salida[VARIABLES].isna().sum().to_string())

reglas = pd.DataFrame({
    "variable": VARIABLES,
    "origen": [
        "pers_genero (1 = Femenino)", "mus_q02: mínimo de horas al día (limpieza)", "amor_pareja (1 = Sí, vacío si no respondió ese formulario)",
        "pers_q10: empieza diciendo que es líder", "pers_q11: primera reacción de molestia/ansiedad",
        "pers_q09: primero menciona pedir ayuda o consultar", "pers_q13: menciona platicar, hablar, convivir...",
        "mus_q04: menciona Spotify", "mus_q04: menciona YouTube o YT",
    ],
})

SALIDA = "Datos_Modelado.xlsx"
with pd.ExcelWriter(SALIDA) as escritor:
    salida.to_excel(escritor, sheet_name="DatosModelado", index=False)
    reglas.to_excel(escritor, sheet_name="Variables", index=False)

print("\nGuardado:", SALIDA, "| personas:", len(salida), "| variables:", len(VARIABLES))
if EN_COLAB:
    files.download(SALIDA)

## Conclusiones

- Con las reglas se construyeron **6 variables nuevas** a partir de texto libre y se prepararon 3 que ya existían, en total **9 variables** para el modelado. Se guardaron en `Datos_Modelado.xlsx`.
- Al revisar persona por persona, las reglas clasificaron las respuestas como esperábamos. Las respuestas que no encajan en ninguna categoría quedaron como **"otro"** (5 en el rol de equipo, 4 en la reacción al cambio y 4 en cómo resuelven un problema): no se forzó ninguna interpretación.
- Las reglas tienen límites, y conviene decirlo: "gana la primera palabra clave", así que una respuesta mixta como *"Investigo y pregunto"* queda en una sola categoría (por su cuenta).
- Faltan datos en dos variables: `tiene_pareja` (4 personas que no respondieron el formulario de relaciones amorosas) y `mus_tiempo_horas` (1 respuesta que no se pudo convertir a número). Se imputarán **dentro** del modelado, solo con los datos de entrenamiento.
- Las preguntas 6 y 7 **no** están entre las variables, porque con ellas se creó el perfil.

**Siguiente cuaderno:** `04_Modelado`, donde entrenamos varios modelos y comparamos sus resultados.